In [1]:
# ============================================================
# Data Source Selector
# ============================================================
# Choose how you're running this notebook, then click "Load Data".
# This sets `dataset_dir` for the rest of the notebook to use.

import ipywidgets as widgets
from IPython.display import display, clear_output
import tensorflow as tf
import pathlib
import os


I0000 00:00:1787456113.836309  898538 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787456114.293923  898538 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787456117.403782  898538 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import tensorflow as tf
import tarfile
import pathlib
import os

DATASET_URL = 'https://raw.githubusercontent.com/RuhanShafi/Tensorflow-Image-Detection/main/Training_Data/Original_faces_catgorized.tgz'

# Path to the archive if the repo was cloned locally (this is what's actually in the repo)
LOCAL_CLONE_ARCHIVE = 'Training_Data/Original_faces_catgorized.tgz'
FOLDER_NAME = 'Original_faces_catgorized'

source_selector = widgets.RadioButtons(
    options=[
        ("I'm running this on Google Colab", "colab"),
        ("I'm running this locally, and only have this notebook (need to download)", "local_download"),
        ("I git cloned the whole repo, including the training data", "local_clone"),
    ],
    description='Setup:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'},
)

load_button = widgets.Button(description="Load Data", button_style='success')
output_area = widgets.Output()

dataset_dir = None  # will be set once the user makes a choice


def extract_archive(archive_path, extract_to):
    """Extract a .tgz/.tar.gz archive to a target directory."""
    with tarfile.open(archive_path) as tar:
        tar.extractall(path=extract_to)


def on_load_clicked(b):
    global dataset_dir
    with output_area:
        clear_output()
        choice = source_selector.value

        if choice == "colab":
            target_dir = os.path.join('.', FOLDER_NAME)
            if os.path.isdir(target_dir):
                print("Dataset already present — skipping download.")
                dataset_dir = str(pathlib.Path(target_dir).resolve())
            else:
                print("Downloading dataset (Colab environment)...")
                archive = tf.keras.utils.get_file(
                    origin=DATASET_URL, extract=True, cache_dir='.', cache_subdir=''
                )
                dataset_dir = os.path.join(os.path.dirname(archive), FOLDER_NAME)

        elif choice == "local_download":
            cached_path = pathlib.Path.home() / '.keras' / 'datasets' / FOLDER_NAME
            if cached_path.is_dir():
                print("Dataset already present in Keras cache — skipping download.")
                dataset_dir = str(cached_path.resolve())
            else:
                print("Downloading dataset (local machine)...")
                archive = tf.keras.utils.get_file(origin=DATASET_URL, extract=True)
                dataset_dir = os.path.join(os.path.dirname(archive), FOLDER_NAME)

        elif choice == "local_clone":
            extracted_path = pathlib.Path(LOCAL_CLONE_ARCHIVE).parent / FOLDER_NAME
            if extracted_path.is_dir():
                print("Dataset already extracted — going straight to import.")
                dataset_dir = str(extracted_path.resolve())
            else:
                archive_path = pathlib.Path(LOCAL_CLONE_ARCHIVE)
                if not archive_path.is_file():
                    print(f"⚠️  Couldn't find the archive at: {archive_path.resolve()}")
                    print("   Make sure you're running this notebook from the repo root, "
                          "and that Training_Data/ was included in your clone (not skipped "
                          "via Git LFS or a shallow/sparse checkout).")
                    return
                print("Found local archive — extracting (no download needed)...")
                extract_archive(archive_path, extract_to=archive_path.parent)
                dataset_dir = str(extracted_path.resolve())

        if not os.path.isdir(dataset_dir):
            print(f"⚠️  Something went wrong — expected data at: {dataset_dir}")
            return

        print(f"\n✅ Dataset ready at: {dataset_dir}")
        print("Contents:", os.listdir(dataset_dir))


load_button.on_click(on_load_clicked)

display(widgets.VBox([source_selector, load_button, output_area]))